# Phase 2: Tool-Integrated Reasoning (TIR) Loop

**Competition:** AI Mathematical Olympiad — Progress Prize 3  
**Goal:** Build the core inference loop that lets the model write and execute Python code while solving math problems. This is the single biggest accuracy improvement in the entire project.

---

## Why we're here

In notebook 01, we saw that Chain-of-Thought prompting helps but has a hard ceiling. The model still hallucinates arithmetic and loses track of variables across many steps. Here's the fix:

> **Instead of asking the model to *compute*, ask it to *write code* that computes.**

The model becomes a *programmer* solving a math problem, not a *calculator*. Python handles all arithmetic exactly. `sympy` handles all algebra symbolically.

---

## Two flavours of TIR

**Single-shot TIR** (simpler):
```
Problem → Model generates [reasoning + code + predicted output + \boxed{answer}]
                                        ↓
                         We execute the code to verify
                                        ↓
                              Return \boxed{answer}
```
The model writes the full solution in one go, including its prediction of what the code will output. We trust `\boxed{}` as the answer.

**Iterative TIR** (more powerful — what we build here):
```
Problem
  → Model generates reasoning + python block
  → WE execute the code → get REAL output
  → Inject real output back into context
  → Model continues reasoning (can self-correct!)
  → Repeat until \boxed{answer}
```
The model reads *real* execution results, not its own predictions. This enables self-correction when the code output surprises the model's expectation.

**Why iterative is better:** The model still hallucinates what the code output will be. Injecting the real output grounds subsequent reasoning in actual computed values.

---

In [ ]:
!pip install -q transformers bitsandbytes accelerate sympy

---

## Section 1: sympy — the model's secret weapon

Before building the loop, understand *why* code execution is so powerful. The key library is **`sympy`** — a Python library for symbolic mathematics.

**Symbolic vs numeric:** A calculator computes `sqrt(2) ≈ 1.41421...`. Sympy keeps it as `√2` — an exact symbolic object — and manipulates it algebraically without any floating-point error.

Olympiad math requires exact answers (integers, fractions, expressions). Sympy delivers this. The model doesn't need to know the answer — it just needs to know how to *express the problem in sympy code*.

In [ ]:
# Demo 1: Algebra — the kind of setup the model writes for olympiad problems
from sympy import symbols, solve, expand, simplify, Eq

# Problem 1 from reference.csv: x+y=10, xy=20, find x^2+y^2
x, y = symbols('x y')

# Method A: direct identity — (x+y)^2 - 2xy
x_plus_y = 10
xy = 20
result_A = x_plus_y**2 - 2*xy
print(f"Method A (identity): x^2 + y^2 = {result_A}")

# Method B: solve the system symbolically
solutions = solve([Eq(x + y, 10), Eq(x * y, 20)], [x, y])
x_val, y_val = solutions[0]
result_B = expand(x_val**2 + y_val**2)
print(f"Method B (symbolic solve): x^2 + y^2 = {result_B}")

# Both give exactly 60 — no floating point, no approximation

In [ ]:
# Demo 2: Combinatorics — exact counting, no arithmetic errors
from sympy import binomial, factorial

# Problem 7 from our hand-crafted set: committees with >= 2 women
# C(10,5) total, minus 0-women, minus 1-woman
total = binomial(10, 5)
zero_women = binomial(6, 5)          # all men
one_woman  = binomial(4, 1) * binomial(6, 4)
at_least_2 = total - zero_women - one_woman

print(f"Total committees: {total}")
print(f"0 women: {zero_women}")
print(f"1 woman: {one_woman}")
print(f">= 2 women: {at_least_2}")   # 186

In [ ]:
# Demo 3: Number theory — the kind of thing olympiad problems demand
from sympy import factorint, isprime, totient, n_order, Rational

# Factorisation (critical for problems involving prime structures)
n = 2025
print(f"factorint({n}) = {factorint(n)}")   # {3: 4, 5: 2} → 3^4 * 5^2

# Euler's totient (appears in number theory olympiad problems)
print(f"phi(2025) = {totient(2025)}")

# Modular arithmetic — modular exponentiation for huge numbers
print(f"3^(2025!) mod 7 = {pow(3, factorial(2025), 7)}")

In [ ]:
# Demo 4: Inclusion-exclusion (number theory problem from reference.csv)
# Count integers 1-1000 divisible by at least one of 3, 5, 7

def count_divisible(n, *divisors):
    """Inclusion-exclusion for multiple divisors."""
    from itertools import combinations
    total = 0
    for r in range(1, len(divisors) + 1):
        for combo in combinations(divisors, r):
            lcm_val = 1
            for d in combo:
                from math import gcd
                lcm_val = lcm_val * d // gcd(lcm_val, d)
            sign = (-1) ** (r + 1)
            total += sign * (n // lcm_val)
    return total

result = count_divisible(1000, 3, 5, 7)
print(f"Count divisible by 3, 5, or 7 in [1,1000]: {result}")  # 543

**Key insight from these demos:** The model doesn't need to know *how* to do number theory — it just needs to know that `sympy.factorint`, `sympy.totient`, `binomial`, etc. exist. That's what its math pretraining provides. We provide the execution environment.

---

## Section 2: Building Blocks

The TIR loop needs three utilities:
1. **Code extractor** — parse `\`\`\`python...\`\`\`` blocks from model output
2. **Safe executor** — run code, capture output, handle errors
3. **Answer extractor** — parse `\boxed{...}` from model output

In [ ]:
import re

def extract_python_blocks(text: str) -> list[str]:
    """
    Extract all ```python...``` code blocks from model output.
    
    Models are trained to put executable code inside these delimiters.
    We extract each block as a separate string to execute in sequence.
    """
    # re.DOTALL makes . match newlines too — needed for multi-line code
    pattern = r"```python\n(.*?)```"
    blocks = re.findall(pattern, text, re.DOTALL)
    return [b.strip() for b in blocks if b.strip()]


# Test it
sample_output = """
Let me solve this step by step.

First, I'll compute x^2 + y^2:
```python
result = 10**2 - 2*20
print(result)
```

So the answer is 60.
"""

blocks = extract_python_blocks(sample_output)
print(f"Found {len(blocks)} code block(s):")
for i, b in enumerate(blocks):
    print(f"  Block {i+1}:")
    print("   ", b.replace('\n', '\n    '))

In [ ]:
import io
import signal
import traceback
from contextlib import redirect_stdout

class ExecutionTimeout(Exception):
    pass

def _timeout_handler(signum, frame):
    raise ExecutionTimeout("Code execution timed out after limit")


def safe_execute(code: str, namespace: dict, timeout_seconds: int = 10) -> str:
    """
    Execute Python code safely and return stdout as a string.
    
    Key design decisions:
    - `namespace` is shared across calls so variables persist between blocks
      (e.g., block 1 defines `x`, block 2 can use `x`)
    - stdout is captured so print() output becomes the 'result'
    - Exceptions return the traceback — the model can read this and self-correct
    - Timeout kills infinite loops (infinite recursion, while True, etc.)
    
    Note: signal.SIGALRM is Unix-only. On Windows, remove the signal lines.
    """
    stdout_buf = io.StringIO()
    
    # Set timeout alarm (Unix/Linux/macOS only)
    signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(timeout_seconds)
    
    try:
        with redirect_stdout(stdout_buf):
            exec(code, namespace)
        output = stdout_buf.getvalue().strip()
        return output if output else "(no output)"
    
    except ExecutionTimeout:
        return f"Error: Code timed out after {timeout_seconds} seconds"
    
    except Exception:
        # Return the full traceback — the model uses this to self-correct
        return f"Error:\n{traceback.format_exc().strip()}"
    
    finally:
        signal.alarm(0)  # always cancel the alarm


# Test: normal execution
ns = {}
print("Normal:", safe_execute("x = 10**2 - 2*20\nprint(x)", ns))

# Test: variable persists across calls
print("Persistence:", safe_execute("print(x + 5)", ns))  # x=60 from above → 65

# Test: error returns traceback
print("Error handling:\n", safe_execute("print(1 / 0)", ns))

In [ ]:
def extract_boxed_answer(text: str) -> int | None:
    """
    Extract the integer inside \\boxed{...} from model output.
    
    Olympiad convention: final answer goes in \\boxed{}.
    Qwen2.5-Math is trained to follow this convention.
    
    Handles common LaTeX wrappers: \\boxed{123}, \\boxed{\\text{123}},
    \\boxed{1{,}234} (thousands separator), etc.
    """
    # Find all \boxed{...} occurrences — take the last one
    # (models sometimes write intermediate \boxed{} before the final answer)
    matches = re.findall(r"\\boxed\{([^}]*)\}", text)
    if not matches:
        return None
    
    raw = matches[-1].strip()
    # Remove LaTeX formatting: \text{x} → x, commas, spaces
    raw = re.sub(r"\\[a-zA-Z]+\{([^}]*)\}", r"\1", raw)  # \text{x} → x
    raw = raw.replace(",", "").replace(" ", "").replace("{", "").replace("}", "")
    
    numbers = re.findall(r"-?\d+", raw)
    return int(numbers[-1]) if numbers else None


def extract_last_integer(text: str) -> int | None:
    """
    Fallback: extract the last integer that appears in the text.
    Used when the model fails to produce \\boxed{}.
    """
    numbers = re.findall(r"-?\d+", text.replace(",", ""))
    return int(numbers[-1]) if numbers else None


# Test
print(extract_boxed_answer(r"Therefore $x^2+y^2 = \boxed{60}$."))          # 60
print(extract_boxed_answer(r"The answer is \boxed{\text{21818}}."))         # 21818
print(extract_boxed_answer(r"We get \boxed{1{,}116} after calculation."))   # 1116
print(extract_boxed_answer("No boxed answer here"))                          # None

---

## Section 3: The Inference Helper (updated for multi-turn)

In notebook 01, `generate_response` took a single prompt string. For TIR we need to pass a **list of messages** (the full conversation history), because the TIR loop adds rounds to the conversation.

We also add `stop_sequences` — generation stops when the model closes a code block with ` ``` `. This enables mid-generation stopping for the iterative loop.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-Math-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print("Model loaded.")

In [ ]:
def generate_from_messages(
    messages: list[dict],
    max_new_tokens: int = 1024,
    temperature: float = 0.0,
) -> str:
    """
    Generate a response from a list of chat messages.
    
    Unlike the single-prompt version in notebook 01, this accepts the full
    conversation history — essential for the multi-round TIR loop where
    each round appends model output + code execution results.
    """
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=temperature if temperature > 0 else None,
            pad_token_id=tokenizer.eos_token_id,
        )

    generated_ids = outputs[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True)

---

## Section 4: The TIR Loop

### The system prompt — why it matters

Qwen2.5-Math-7B-Instruct was trained with a specific TIR system prompt. Using it tells the model:
- Write reasoning in natural language
- Put executable code in ` ```python ` blocks
- Put the final answer in `\boxed{}`

Without this system prompt, the model may write reasoning only (CoT mode) instead of code. The right system prompt is the most important hyperparameter in TIR.

### The loop structure

```
Round 1:
  [system: TIR prompt]
  [user: problem]
  → Model generates: reasoning + ```python block```
  → We execute: get real output
  
Round 2:
  [... history ...]
  [user: ```output\nreal output\n```]
  → Model generates: more reasoning OR \boxed{answer}
  → If \boxed{}: done. Else: repeat.
```

The `user: output` injection is the key move — we slip the real execution result into the conversation as if the model saw it. This is how the model grounds its reasoning in reality.

In [ ]:
# This system prompt is taken from the Qwen2.5-Math paper and activates TIR mode.
# The model was specifically trained to respond to this exact phrasing.
TIR_SYSTEM_PROMPT = (
    "Please integrate natural language reasoning with programs to solve the problem above, "
    "and put your final answer within \\boxed{}."
)


def run_tir_loop(
    problem: str,
    max_rounds: int = 5,
    max_new_tokens: int = 1024,
    temperature: float = 0.0,
    verbose: bool = False,
) -> dict:
    """
    Tool-Integrated Reasoning loop.
    
    Each round:
      1. Generate model response (reasoning + optional python block)
      2. Execute any python code in a shared namespace
      3. Inject real output back as a user message
      4. Repeat until \\boxed{} found or max_rounds reached
    
    Returns a dict with:
      'answer'   : int or None  (the extracted \\boxed{} answer)
      'history'  : list of round dicts (response, code, output, boxed)
      'messages' : full conversation (for inspection / continuation)
    """
    messages = [
        {"role": "system", "content": TIR_SYSTEM_PROMPT},
        {"role": "user",   "content": problem},
    ]
    
    # Shared Python namespace: variables defined in round 1 are available in round 2
    namespace = {}
    history   = []
    final_answer = None

    for round_num in range(max_rounds):
        if verbose:
            print(f"\n{'─'*60}")
            print(f"  ROUND {round_num + 1} / {max_rounds}")
            print(f"{'─'*60}")

        # --- Generate ---
        response = generate_from_messages(
            messages,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )

        if verbose:
            print(response)

        # --- Parse ---
        code_blocks = extract_python_blocks(response)
        boxed       = extract_boxed_answer(response)

        # Append model turn to history
        messages.append({"role": "assistant", "content": response})

        round_record = {
            "round":    round_num + 1,
            "response": response,
            "code":     code_blocks,
            "output":   None,
            "boxed":    boxed,
        }

        if not code_blocks:
            # Pure text round — either final answer or stuck
            if boxed is not None:
                final_answer = boxed
            history.append(round_record)
            break

        # --- Execute all code blocks in sequence ---
        outputs = []
        for code in code_blocks:
            out = safe_execute(code, namespace)
            outputs.append(out)

        combined_output = "\n".join(outputs).strip()
        round_record["output"] = combined_output
        history.append(round_record)

        if verbose:
            print(f"\n>>> Execution output:")
            print(combined_output)

        # If the model already gave \boxed{}, we're done regardless of output
        if boxed is not None:
            final_answer = boxed
            break

        # --- Inject real output back for next round ---
        # This is the key move: we slip real results into the conversation.
        # The model's next response is conditioned on actual computed values.
        messages.append({
            "role":    "user",
            "content": f"```output\n{combined_output}\n```",
        })

    # If \boxed{} never appeared, try extracting any integer from the last response
    if final_answer is None and history:
        final_answer = extract_last_integer(history[-1]["response"])

    return {
        "answer":   final_answer,
        "history":  history,
        "messages": messages,
    }

---

## Section 5: Verbose Walkthrough — One Problem, Step by Step

Before running the full benchmark, watch the loop run on **Problem 1** (Alice and Bob sweets, answer=50). This is the easiest reference problem — if the TIR loop works at all, it should solve this.

Read every round of output carefully:
- What code does the model write?
- What does the real execution return?
- Does the model use that output in round 2?
- Where does `\boxed{}` appear?

In [ ]:
import pandas as pd

REFERENCE_CSV = '../Dataset/reference.csv'
ref_df = pd.read_csv(REFERENCE_CSV)
problems = ref_df.to_dict('records')

# Problem 1: Alice and Bob (answer = 50)
p1 = problems[0]
print(f"Problem: {p1['problem'][:120]}...")
print(f"Expected answer: {p1['answer']}")
print()

result = run_tir_loop(
    problem=p1['problem'],
    max_rounds=5,
    verbose=True,    # show each round in detail
)

print(f"\n{'='*60}")
print(f"TIR extracted answer : {result['answer']}")
print(f"Expected             : {p1['answer']}")
print(f"Correct              : {result['answer'] == p1['answer']}")
print(f"Rounds used          : {len(result['history'])}")

In [ ]:
# Inspect the full conversation that was built up
print("Full conversation structure:")
for i, msg in enumerate(result['messages']):
    role    = msg['role'].upper()
    preview = msg['content'][:120].replace('\n', ' ')
    print(f"  [{i}] {role:10s}: {preview}...")

---

## Section 6: Full Benchmark Run

Run TIR on all 10 reference problems. This will take 10–30 minutes depending on GPU.

**Watch for:**
- Which problems the model solves that CoT didn't (the TIR gain)
- Which problems still fail and why (conceptual vs computational gap)
- How many rounds each problem uses

In [ ]:
difficulty = {
    '92ba6a': ('P1',  'AIMO2', 'Easy'),
    'a295e9': ('P2',  'AIMO2', 'Medium'),
    '0e644e': ('P3',  'AIMO2', 'Hard'),
    '9c1c5f': ('P4',  'AIMO2', 'Very Hard'),
    '424e18': ('P5',  'AIMO3', 'Extreme'),
    '26de63': ('P6',  'AIMO3', 'Extreme'),
    '42d360': ('P7',  'AIMO3', 'Extreme'),
    '641659': ('P8',  'AIMO3', 'Extreme'),
    'dd7f5e': ('P9',  'AIMO3', 'Extreme'),
    '86e8e5': ('P10', 'AIMO3', 'Extreme'),
}

tir_results = []

for p in problems:
    d = difficulty.get(p['id'], ('?', '?', '?'))
    print(f"Running {d[0]} ({d[2]})... ", end='', flush=True)

    result = run_tir_loop(
        problem=p['problem'],
        max_rounds=5,
        max_new_tokens=1024,
        verbose=False,
    )

    predicted = result['answer']
    correct   = (predicted == int(p['answer']))
    rounds    = len(result['history'])

    tir_results.append({
        'id':        p['id'],
        'tier':      d[0],
        'source':    d[1],
        'difficulty':d[2],
        'expected':  int(p['answer']),
        'predicted': predicted,
        'correct':   correct,
        'rounds':    rounds,
        'result':    result,
    })

    status = 'CORRECT ✓' if correct else f'WRONG (got {predicted})'
    print(f"{status}  [{rounds} round(s)]")

n_correct = sum(r['correct'] for r in tir_results)
print(f"\nTIR score: {n_correct} / {len(tir_results)}")

---

## Section 7: Results Comparison — CoT vs TIR

In [ ]:
# CoT scores from notebook 01 — paste your actual results here
# Format: {problem_id: True/False}
# These are placeholders — replace with your actual notebook 01 results
cot_correct = {
    '92ba6a': None,   # P1  - fill in after running notebook 01
    'a295e9': None,   # P2
    '0e644e': None,   # P3
    '9c1c5f': None,   # P4
    '424e18': None,   # P5
    '26de63': None,   # P6
    '42d360': None,   # P7
    '641659': None,   # P8
    'dd7f5e': None,   # P9
    '86e8e5': None,   # P10
}

print(f"{'#':<5} {'Tier':<5} {'Source':<7} {'Expected':<10} {'CoT':<8} {'TIR':<8} {'Rounds':<7} {'Gain?'}")
print("─" * 65)

for r in tir_results:
    cot = cot_correct.get(r['id'])
    cot_str = '✓' if cot is True else ('✗' if cot is False else '?')
    tir_str = '✓' if r['correct'] else '✗'
    gain    = '↑ gained' if (cot is False and r['correct']) else (
              '↓ lost'   if (cot is True  and not r['correct']) else '')
    print(f"{r['tier']:<5} {r['source']:<7} {r['expected']:<10} {cot_str:<8} {tir_str:<8} {r['rounds']:<7} {gain}")

print("─" * 65)
tir_total = sum(r['correct'] for r in tir_results)
cot_total = sum(1 for v in cot_correct.values() if v is True)
print(f"{'TOTAL':<13} {'':10} {cot_total}/10    {tir_total}/10")
print()
print("Reference: best open model (gpt-oss-120B) = 4/10")

---

## Section 8: Failure Analysis

For every problem TIR still gets wrong, read the full conversation history. Classify the failure:

| Failure type | Description | Fix in next notebook |
|---|---|---|
| **Wrong approach** | Model set up the wrong equation/framework | Better prompting, bigger model |
| **Code error** | Code raised an exception, model couldn't recover | More rounds, better error handling |
| **Right code, wrong answer** | Code ran but gave a wrong number | Bug in code the model wrote |
| **No boxed** | Model gave up without a final answer | Higher `max_rounds` or `max_new_tokens` |
| **Conceptual** | Problem requires insight no 7B model has | Bigger model or fine-tuning |

In [ ]:
failures = [r for r in tir_results if not r['correct']]

if not failures:
    print("No failures! Exceptional result — move to notebook 03 (majority voting).")
else:
    print(f"{len(failures)} failure(s):\n")

for r in failures:
    print(f"{'='*70}")
    print(f"Problem {r['tier']} ({r['difficulty']}) | Expected: {r['expected']} | Got: {r['predicted']}")
    print()

    for h in r['result']['history']:
        print(f"  --- Round {h['round']} ---")
        print(f"  Response (first 400 chars):")
        print("  " + h['response'][:400].replace('\n', '\n  '))

        if h['code']:
            print(f"\n  Code (first block, first 200 chars):")
            print("  " + h['code'][0][:200].replace('\n', '\n  '))

        if h['output']:
            print(f"\n  Output:")
            print("  " + h['output'][:300].replace('\n', '\n  '))

        print(f"  Boxed answer extracted: {h['boxed']}")
        print()

---

## What Did We Learn?

### Key observations (fill in after running)

1. **TIR vs CoT gap:** TIR improved accuracy by ___ problems over the best CoT result.  
   The problems where TIR gained: ___. This tells you what CoT's ceiling was.

2. **Where the gain comes from:** For problems TIR solved that CoT didn't, look at the code the model wrote.  
   Was it using `sympy.solve`? Brute-force enumeration? `factorint`?  
   The model's insight was right — it just needed Python to compute it correctly.

3. **The round structure:** Most problems should complete in 1–2 rounds.  
   If a problem uses all 5 rounds, it's either making progress (each round corrects an error)  
   or looping (each round makes the same mistake).

4. **The AIMO2 divide:** Problems P1–P4 are where a 7B model can realistically contribute.  
   P5–P10 require mathematical insight beyond what 7B parameters can reliably provide,  
   regardless of how well the inference loop is built. The fix is a bigger model (Phase 4).

5. **Self-correction in action:** Find a round where the code produced an error (traceback).  
   Did the model fix the bug in the next round? This is the most impressive TIR behaviour.

### What comes next (Notebook 03)

TIR with a single sample is still stochastic — with `temperature=0` (greedy) the model always  
gives the same answer, but with `temperature>0` answers vary. **Majority voting** exploits  
this: sample TIR 10–48 times, take the most common answer.

This is Phase 3 (Self-Consistency) and it's responsible for a large fraction of the gap  
between single-run accuracy and competition-level accuracy.

---

## Exercises

1. **Increase `max_rounds` to 8** for the hardest problem that failed.  
   Does more room to self-correct help, or does the model loop?

2. **Read the PDF solution for a problem TIR got wrong** (`Dataset/AIMO3_Reference_Problems.pdf`).  
   Then look at what code the model wrote. At what step did the model's approach diverge  
   from the correct proof? Was the divergence fixable with better code?

3. **Try removing the TIR system prompt** — just pass the problem with no system message.  
   Does the model still write code? How does accuracy change?  
   (Answer: it usually falls back to CoT mode — proving the system prompt is load-bearing.)

4. **Inspect the namespace** after a successful run:  
   ```python
   result = run_tir_loop(problems[0]['problem'], verbose=False)
   # Then look at what variables the model created:
   ns = {}  # passed inside the loop
   ```
   Understanding what the model puts in the namespace reveals its problem-solving strategy.

5. **Extend the loop for error recovery:** After a code error, inject a hint:  
   ```python
   if "Error" in combined_output:
       messages.append({"role": "user", "content": 
           f"```output\n{combined_output}\n```\nPlease fix the error and try again."})
   ```
   Does explicit error prompting help the model recover?